# Notebook 03 — Phase 1B Model Comparison Skeleton

This is the Phase 1B model comparison skeleton for LSTM, TCN, and standard DLinear baselines under no-trade-band binary labels.

Default mode is smoke-only: no training, no checkpoint writing, and no result artifact writing. Full Colab execution is deferred to a later explicit step.

In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd

FULL_RUN = False
RUN_TRAINING = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path("/content/drive/MyDrive/stockdata")
RAW_DATA_DIR = DATA_ROOT / "Dow_30_1min"
OUTPUT_DIR = DATA_ROOT / "phase1b_notebook03_model_comparison

In [ ]:
from ml_utils.metrics import always_predict_baseline_metrics
from ml_utils.metrics import compute_classification_metrics
from ml_utils.metrics import dummy_baseline_metrics
from ml_utils.metrics import format_metrics_table
from ml_utils.models.lstm_classifier import LSTMClassifier
from ml_utils.models.tcn_classifier import TCNClassifier
from ml_utils.models.dlinear_classifier import DLinearClassifier

In [ ]:
TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
SEEDS = [42, 43, 44]

WINDOW_SIZE = 12
LABEL_HORIZON_K = 12
THRESHOLD_BPS = 5

NUM_CLASSES = 2
INPUT_SIZE = 12

BATCH_SIZE = 512
LEARNING_RATE = 1e-4
MAX_EPOCHS = 20
EARLY_STOP_PATIENCE = 5

## Historical LSTM Context

P1B.10 Candidate A single-seed result:

- `model_macro_f1_mean` ≈ 0.5203
- `dummy_stratified_macro_f1_mean` ≈ 0.4990
- delta ≈ +0.0213

P1B.11b strict multi-seed result:

- seed 42 delta ≈ +0.0213
- seed 43 delta ≈ +0.0038
- seed 44 delta ≈ -0.0089
- overall mean delta ≈ +0.0054
- std ≈ 0.0152

Interpretation: LSTM is weak, seed-sensitive, and not robustly better than dummy. These values are historical context, not Notebook 03 output.

In [ ]:
MODEL_REGISTRY = [
    {
        "model_name": "lstm",
        "import_path": "ml_utils.models.lstm_classifier.LSTMClassifier",
        "constructor": LSTMClassifier,
        "config": {
            "input_size": INPUT_SIZE,
            "hidden_size": 64,
            "num_layers": 2,
            "dropout": 0.2,
            "num_classes": NUM_CLASSES,
        },
    },
    {
        "model_name": "tcn",
        "import_path": "ml_utils.models.tcn_classifier.TCNClassifier",
        "constructor": TCNClassifier,
        "config": {
            "input_size": INPUT_SIZE,
            "num_channels": [32, 32],
            "kernel_size": 3,
            "dropout": 0.1,
            "num_classes": NUM_CLASSES,
            "causal": True,
        },
    },
    {
        "model_name": "dlinear",
        "import_path": "ml_utils.models.dlinear_classifier.DLinearClassifier",
        "constructor": DLinearClassifier,
        "config": {
            "seq_len": WINDOW_SIZE,
            "input_size": INPUT_SIZE,
            "num_classes": NUM_CLASSES,
            "moving_avg_kernel": 5,
            "individual": False,
        },
    },
]

model_registry_df = pd.DataFrame(
    [
        {
            "model_name": item["model_name"],
            "import_path": item["import_path"],
            "config": item["config"],
        }
        for item in MODEL_REGISTRY
    ]
)
model_registry_df

In [ ]:
NOTEBOOK03_RESULT_COLUMNS = [
    "model_name",
    "ticker",
    "seed",
    "window_size",
    "label_horizon_k",
    "threshold_bps",
    "split",
    "n_train_windows",
    "n_val_windows",
    "n_test_windows",
    "label_retained_pct",
    "model_macro_f1",
    "model_balanced_accuracy",
    "dummy_stratified_macro_f1_mean",
    "dummy_stratified_macro_f1_std",
    "dummy_prior_macro_f1",
    "always_up_macro_f1",
    "always_down_macro_f1",
    "delta_macro_f1_vs_dummy",
    "confusion_matrix",
]

results_schema_df = pd.DataFrame(columns=NOTEBOOK03_RESULT_COLUMNS)
results_schema_df

In [ ]:
ARTIFACT_NAMES = {
    "per_ticker_results": "notebook03_per_ticker_results.csv",
    "summary_by_model": "notebook03_summary_by_model.csv",
    "summary_by_seed": "notebook03_summary_by_seed.csv",
    "run_manifest": "notebook03_run_manifest.json",
}

RUN_MANIFEST_TEMPLATE = {
    "notebook": "03_model_comparison.ipynb",
    "phase": "P1B.15",
    "git_commit_hash": None,
    "full_run": FULL_RUN,
    "run_training": RUN_TRAINING,
    "models": ["lstm", "tcn", "dlinear"],
    "tickers": TICKERS,
    "seeds": SEEDS,
    "window_size": WINDOW_SIZE,
    "label_horizon_k": LABEL_HORIZON_K,
    "threshold_bps": THRESHOLD_BPS,
}

artifact_manifest_df = pd.DataFrame(
    [{"artifact_key": key, "filename": value} for key, value in ARTIFACT_NAMES.items()]
)
artifact_manifest_df

In [ ]:
def require_training_enabled() -> None:
    if not RUN_TRAINING:
        raise RuntimeError(
            "RUN_TRAINING is False. Notebook 03 is currently skeleton/smoke-only. "
            "Enable RUN_TRAINING only in a later explicit full-run step."
        )


def run_model_comparison():
    require_training_enabled()
    raise NotImplementedError(
        "Full model comparison execution is deferred to a later explicit step."
    )

In [ ]:
import torch

smoke_x = torch.randn(2, WINDOW_SIZE, INPUT_SIZE)
smoke_rows = []
for registry_item in MODEL_REGISTRY:
    model = registry_item["constructor"](**registry_item["config"])
    model.eval()
    with torch.no_grad():
        logits = model(smoke_x)
    output_shape = tuple(logits.shape)
    assert output_shape == (2, NUM_CLASSES), (
        f"{registry_item['model_name']} returned {output_shape}, "
        f"expected {(2, NUM_CLASSES)}"
    )
    smoke_rows.append(
        {
            "model_name": registry_item["model_name"],
            "output_shape": output_shape,
        }
    )

smoke_shapes_df = pd.DataFrame(smoke_rows)
smoke_shapes_df

## Final Note

This notebook is intentionally not executed as a full run. The next step should be review plus atomic commit of the skeleton. Full Colab training requires a separate explicit prompt.